In [5]:
%load_ext autoreload
%autoreload 2
import os
import sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = '1'
import numpy as np
import pandas as pd
from typing import Dict, List
# from dotenv import load_dotenv  # pip install python-dotenv
from dynclim.features.safety import SegmentationConfig, ThresholdConfig, CompositeConfig 
from dynclim.experiments.runner import ExperimentRunner
from dynclim.data import PHMSADataPipeline
from dynclim.utils import loadDomainsFromYaml
import arrow
# from dynclim.data.database import get_reports_from_db, create_database_engine
# load_dotenv(dotenv_path='/dynClimToolkit/.env')
# load_dotenv()
from pathlib import Path
# If your notebook is in the project root
# project_root = Path("/Users/mrosen44/Documents/dev/DynamicNarrativeClimateToolkit")
project_root = Path("/Users/michaelrosen/Documents/dev/DynamicNarrativeClimateToolkit")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
# prepare domains and data for feature extraction
domain_definitions = loadDomainsFromYaml(project_root / 'config/domains/safety_domains.yaml')

# clean and prep reports for feature generation
pipeline = PHMSADataPipeline(
    base_data_dir=project_root / "dynclim/data/raw/phmsa/accident_incident_data",
    output_dir=project_root / "dynclim/data/processed/phmsa",
    include_historical=True
)
results = pipeline.run()
report_data = pipeline.get_report_data()  # [(eid, narrative), ...]

2026-04-28 13:44:03,230 - INFO - Loading modern/hazardous_liquid from accident_hazardous_liquid_jan2010_present.txt
2026-04-28 13:44:03,654 - INFO -   5,890 rows, 648 columns
2026-04-28 13:44:04,199 - INFO - Loading modern/gas_transmission from incident_gas_transmission_gathering_jan2010_present.txt
2026-04-28 13:44:04,327 - INFO -   2,002 rows, 624 columns
2026-04-28 13:44:04,548 - INFO - Loading modern/gas_distribution from incident_gas_distribution_jan2010_present.txt
2026-04-28 13:44:04,637 - INFO -   1,560 rows, 468 columns
2026-04-28 13:44:04,792 - INFO - Loading modern/gravity_regulated from accident_gravity_reporting_regulated_jul2020_present.txt
2026-04-28 13:44:04,812 - INFO -   540 rows, 138 columns
2026-04-28 13:44:04,860 - INFO - Loading era3/hazardous_liquid_2002_2009 from accident_hazardous_liquid_jan2002_dec2009.txt
2026-04-28 13:44:04,927 - INFO -   3,030 rows, 259 columns
2026-04-28 13:44:05,077 - INFO - Loading era3/gas_transmission_2002_2009 from incident_gas_transm

In [7]:
# Run experiment with checkpointing

runner = ExperimentRunner(
    domain_definitions = domain_definitions,
    use_acceleration=True,
    device_pref='auto',
    use_parallel=False,
    embedding_model_names=[      
                                 
        'all-MiniLM-L6-v2',        # small, from sentence transformer team
        # 'all-mpnet-base-v2',     # medium, from sentence transformer team
        'all-roberta-large-v1',  # large, from sentence transformer team
        # 'BAAI/bge-large-en-v1.5' # very large, and generally better performance, but has different training method
                                   # when compared to above models which vary just on size
        # 'mixedbread-ai/mxbai-embed-large-v1',   # mid-sized
        'BAAI/bge-m3'                           # successor to BAAI model above
        ]
)

# Test with subset of configurations
config_filter = {
    #'segmentation_methods': ['sentence'],
    #'threshold_methods': ['top_k'],
    #'composite_methods': ['attention_weight', 'weighted_average'],
    'sentiment_methods': ['vader','roberta','siebert'] #['roberta', 'distilbert', 'finbert','vader','siebert']
}

results_df = runner.run_experiment(
    report_data, 
    experiment_name = "phmsa_04-28-2026",
    config_filter=config_filter,
    resume_from_checkpoint=True,
    # save_interval=2,
    chunk_size=None
    # max_workers=10
)

   Registry: dropped 25 constant columns → 9 remaining (constants in registry_constants.json)

🔬 Running experiment 'phmsa_04-28-2026'
   Total configurations: 135
   Pending configurations: 135
   Reports: 19925
   Embedding models: 3
   Parallel processing: False
   Caching: Enabled (embeddings=True, segments=True)


Configurations:   0%|          | 0/135 [00:00<?, ?it/s]


🔄 Initializing processor:
   Embedding model: BAAI/bge-m3
   Sentiment method: roberta
🚀 Using Apple Silicon MPS acceleration


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
2026-04-28 13:44:45,043 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Initialized Processor:
   Domains: ['LSC', 'SC', 'ST', 'CSP', 'SEH', 'SI', 'SR']
   Total items: 30
   Device: mps
   Caching: embeddings=True, segments=True
   Injected sentiment cache: 0 entries

🚀 Using batch-optimized processing...
   Pre-segmenting all texts...
[seg] SentenceSegmenter backend=spacy, min_length=5, unit=chars
[seg] built segmenter SentenceSegmenter for key=('5a5a91d711baaba23988779567c0b9f2', 'chars')
   Found 163059 unique segments
   Pre-encoding unique segments...
   ✅ Pre-cached 163059 embeddings

🚀 Starting processing...
   Reports: 19925
   Parallel: False


... (284 outputs left out)

Configurations:  99%|█████████▉| 134/135 [8:52:06<02:03, 123.88s/it, cache_hit=99.0%, time=129.2s]

✅ Processing complete!
   Total time: 128.46 seconds
   Rate: 155.1 reports/second

💾 Checkpoint saved: 134 configurations
   Full results directory: checkpoints/phmsa_04-28-2026/_cfg

🚀 Using batch-optimized processing...
   Pre-segmenting all texts...
   Found 163059 unique segments
   Pre-encoding unique segments...
   ✅ Pre-cached 163059 embeddings

🚀 Starting processing...
   Reports: 19925
   Parallel: False


Configurations: 100%|██████████| 135/135 [8:52:47<00:00, 236.80s/it, cache_hit=99.1%, time=41.3s] 

✅ Processing complete!
   Total time: 40.53 seconds
   Rate: 491.6 reports/second

💾 Checkpoint saved: 135 configurations
   Full results directory: checkpoints/phmsa_04-28-2026/_cfg

💾 Checkpoint saved: 135 configurations
   Full results directory: checkpoints/phmsa_04-28-2026/_cfg

✅ Experiment complete!
   Total time: 31967.8 seconds
   Average time per config: 236.80 seconds

📊 Final Cache Statistics:

📊 Cache Performance Report
Device: mps

🎯 Hit Rates:
   Segment Embeddings: 98.9%
   Sentiments: 100.0%
   Text Segmentation: 98.9%
   Similarities: 0.0%
   Overall: 99.1%

📈 Reuse Statistics:
   Unique segments: 163,054
   Total processed: 15,005,295
   Reuse factor: 92.03x

💾 Cache Sizes:
   segment_embeddings: 163,054 entries
   sentiments: 163,054 entries
   text_segments: 19,826 entries
   similarities: 0 entries
📄 Results exported to: checkpoints/phmsa_04-28-2026_results.csv
📄 Results exported to: checkpoints/phmsa_04-28-2026_results.h5

📁 Full report-level results saved in: ch